# 518. Coin Change II

## Topic Alignment
- Counting combinations appears in ensemble methods (counting valid model configurations), hyperparameter tuning (counting parameter combinations), and resource allocation scenarios in distributed systems.

## Metadata Summary
- Source: https://leetcode.com/problems/coin-change-ii/
- Tags: Dynamic Programming, Array, Complete Knapsack, Counting
- Difficulty: Medium
- Priority: High

## Problem Statement
You are given an integer array `coins` representing coins of different denominations and an integer `amount` representing a total amount of money.

Return the **number of combinations** that make up that amount. If that amount of money cannot be made up by any combination of the coins, return `0`.

You may assume that you have an **infinite number** of each kind of coin.

The answer is **guaranteed** to fit into a signed **32-bit** integer.

**Constraints**:
- 1 <= coins.length <= 300
- 1 <= coins[i] <= 5000
- All the values of coins are unique
- 0 <= amount <= 5000

## Progressive Hints
- Hint 1: This is a complete knapsack problem focused on counting combinations, not permutations.
- Hint 2: Use dp[i] to represent the number of ways to make amount i.
- Hint 3: For combinations (not permutations), iterate coins in outer loop, amounts in inner loop.
- Hint 4: For each amount, add the ways from (amount - coin): dp[amt] += dp[amt - coin].
- Hint 5: Initialize dp[0] = 1 (one way to make 0: use no coins).

## Solution Overview
This is a **Complete Knapsack** counting problem with a critical distinction:
- We count **combinations**, not permutations
- [1,2] and [2,1] are the same combination
- Different from LC 377 which counts permutations

**Key Insight - Loop Order**:
- **Combinations**: Outer loop = coins, Inner loop = amounts
- **Permutations**: Outer loop = amounts, Inner loop = coins

**State**: `dp[i]` = number of combinations to make amount i

**Recurrence**: 
```python
dp[amt] += dp[amt - coin]  # Add ways from previous amount
```

## Detailed Explanation

### Combinations vs Permutations

**Example**: coins = [1, 2], amount = 3

**Combinations** (this problem):
- [1, 1, 1]
- [1, 2]
- Total: **2 combinations**

**Permutations** (LC 377):
- [1, 1, 1]
- [1, 2]
- [2, 1]
- Total: **3 permutations**

---

### Loop Order Determines Combinations vs Permutations

**Combinations (this problem) - Outer loop: coins**:
```python
for coin in coins:              # Outer: iterate coins
    for amt in range(coin, amount + 1):  # Inner: iterate amounts
        dp[amt] += dp[amt - coin]
```
- **Why this works**: When processing coin C, we only consider combinations that haven't used coins after C yet
- This ensures each combination is counted exactly once
- Example: For coins [1,2], when we process coin=2, we add to amounts using combinations from coin=1
- [1,2] is counted, but [2,1] is not (because when processing coin=1, we haven't added coin=2's contributions yet)

**Permutations (LC 377) - Outer loop: amounts**:
```python
for amt in range(1, amount + 1):  # Outer: iterate amounts
    for coin in coins:             # Inner: iterate coins
        if amt >= coin:
            dp[amt] += dp[amt - coin]
```
- **Why this works**: For each amount, we try all coins
- This counts different orderings separately
- Example: [1,2] and [2,1] are both counted

---

### Why Does Loop Order Matter?

**Combinations (coins outer)**:
- Process coins one by one in a fixed order
- For coin C, only use combinations built from coins {1, 2, ..., C-1}
- Ensures we don't count same combination multiple times

**Permutations (amounts outer)**:
- For each amount, try all coins equally
- No fixed coin order
- Same set of coins in different orders counted separately

---

### Example Walkthrough

**Input**: coins = [1, 2, 5], amount = 5

**Initial**: `dp = [1, 0, 0, 0, 0, 0]` (dp[0]=1: one way to make 0)

**After coin = 1**:
- dp[1] += dp[0] = 1  (way: [1])
- dp[2] += dp[1] = 1  (way: [1,1])
- dp[3] += dp[2] = 1  (way: [1,1,1])
- dp[4] += dp[3] = 1  (way: [1,1,1,1])
- dp[5] += dp[4] = 1  (way: [1,1,1,1,1])
- `dp = [1, 1, 1, 1, 1, 1]`

**After coin = 2**:
- dp[2] += dp[0] = 1+1 = 2  (ways: [1,1], [2])
- dp[3] += dp[1] = 1+1 = 2  (ways: [1,1,1], [1,2])
- dp[4] += dp[2] = 1+2 = 3  (ways: [1,1,1,1], [1,1,2], [2,2])
- dp[5] += dp[3] = 1+2 = 3  (ways: [1,1,1,1,1], [1,1,1,2], [1,2,2])
- `dp = [1, 1, 2, 2, 3, 3]`

**After coin = 5**:
- dp[5] += dp[0] = 3+1 = 4  (add way: [5])
- `dp = [1, 1, 2, 2, 3, 4]`

**Result**: dp[5] = 4

**The 4 combinations are**:
1. [1, 1, 1, 1, 1]
2. [1, 1, 1, 2]
3. [1, 2, 2]
4. [5]

---

### Complete Knapsack Template Review

**For Combinations** (this problem):
```python
for coin in coins:                      # Outer: coins
    for amt in range(coin, amount + 1): # Inner: amounts (left to right)
        dp[amt] += dp[amt - coin]
```

**For Permutations** (LC 377):
```python
for amt in range(1, amount + 1):        # Outer: amounts
    for coin in coins:                  # Inner: coins
        if amt >= coin:
            dp[amt] += dp[amt - coin]
```

**Both are complete knapsack** (left to right traversal), but **loop order** determines combinations vs permutations.

## Complexity Trade-off Table
| Approach | Time | Space | Notes |
| --- | --- | --- | --- |
| Backtracking | O(amount^n) | O(amount) | Too slow |
| Top-down DP + memo | O(n × amount) | O(n × amount) | Clearer logic |
| Bottom-up DP | O(n × amount) | O(amount) | Most efficient |
| 2D DP | O(n × amount) | O(n × amount) | Easier to understand |

In [ ]:
class Solution:
    def change(self, amount: int, coins: list[int]) -> int:
        """
        Complete Knapsack - counting combinations.
        
        Time: O(n × amount)
        Space: O(amount)
        """
        # dp[i] = number of combinations to make amount i
        dp = [0] * (amount + 1)
        dp[0] = 1  # One way to make 0: use no coins
        
        # IMPORTANT: Outer loop = coins, Inner loop = amounts
        # This gives COMBINATIONS (not permutations)
        for coin in coins:
            # Complete knapsack: left to right
            for amt in range(coin, amount + 1):
                # Add number of ways to make (amt - coin)
                dp[amt] += dp[amt - coin]
        
        return dp[amount]

In [ ]:
# Test cases
tests = [
    (5, [1, 2, 5], 4),         # [1,1,1,1,1], [1,1,1,2], [1,2,2], [5]
    (3, [2], 0),               # Impossible
    (10, [10], 1),             # Exactly one coin
    (0, [1], 1),               # Amount 0
    (4, [1, 2, 3], 4),         # [1,1,1,1], [1,1,2], [2,2], [1,3]
    (5, [1, 2, 3], 5),         # Multiple combinations
    (500, [3, 5, 7, 8, 9, 10, 11], 35502874),  # Large case
]

solver = Solution()
for amount, coins, expected in tests:
    result = solver.change(amount, coins)
    assert result == expected, f"Failed for amount={amount}, coins={coins}: got {result}, expected {expected}"
print('All tests passed.')

## Complexity Analysis
- **Time**: O(n × amount) where n = len(coins)
  - For each coin, iterate through amounts from coin to target
  - Total iterations: n × amount
- **Space**: O(amount)
  - Single DP array of size amount + 1
  - Can't optimize further without changing algorithm

## Edge Cases & Pitfalls
- **amount = 0**: Return 1 (one way: use no coins)
- **Empty coins array**: By constraints won't happen, but would return 0 for amount > 0
- **Single coin**: If amount % coin == 0, return 1; else return 0
- **Loop order**: CRITICAL! Outer=coins for combinations, outer=amounts for permutations
- **Initialization**: dp[0] = 1 is essential
- **Integer overflow**: Problem guarantees answer fits in 32-bit int
- **Large values**: With constraints (amount ≤ 5000, coins ≤ 300), DP is feasible
- **Comparison with LC 377**: Same template, different loop order!

## Follow-up Variants
- **Count permutations**: See LC 377 Combination Sum IV (swap loop order)
- **Limited coins**: Each coin has limited quantity (becomes bounded knapsack)
- **Weighted combinations**: Each coin has different value, count combinations by value
- **Minimum/maximum coins**: Find combination with min/max number of coins
- **Print all combinations**: Reconstruct actual combinations, not just count
- **Multiple targets**: Count combinations for all amounts from 1 to N

## Takeaways
- **Loop Order is Critical**: Outer=coins gives combinations, outer=amounts gives permutations
- **Combinations Pattern**: Process items in outer loop to avoid counting duplicates
- **Complete Knapsack**: Traverse left to right to allow unlimited item reuse
- **Counting DP**: Use `dp[amt] += dp[amt-coin]` to accumulate ways
- **Base Case**: dp[0] = 1 represents one way to make zero
- **Template Reusability**: Same DP structure as LC 322, but counting instead of minimizing
- **Compare with LC 377**: Understanding the loop order difference is key to mastering both problems

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| LC 322 | Coin Change | Complete knapsack minimization |
| LC 377 | Combination Sum IV | Complete knapsack permutations |
| LC 279 | Perfect Squares | Complete knapsack minimization |
| LC 39 | Combination Sum | Backtracking + DP |
| LC 494 | Target Sum | 0/1 knapsack counting |